# VaR Historico e Filtered Historical Simulation (FHS)

Neste notebook, exploramos metodos **nao-parametricos** e **semi-parametricos** de VaR
que nao assumem uma distribuicao especifica para os retornos.

**VaR Historico**: usa diretamente os quantis empiricos da distribuicao passada dos retornos.

**Filtered Historical Simulation (FHS)**: combina o melhor de dois mundos — a dinamica
temporal do GARCH com a flexibilidade da distribuicao empirica dos residuos padronizados.

**Conteudo:**
1. VaR Historico (rolling window)
2. Problemas do VaR historico
3. Filtered Historical Simulation (FHS) - Barone-Adesi et al. (1999)
4. FHS passo a passo
5. Comparacao: Historico vs FHS vs Parametrico

**Referencias:**
- Barone-Adesi, G., Bourgoin, F., & Giannopoulos, K. (1998). Don't look back. *Risk*, 11, 100-103.
- Barone-Adesi, G., Giannopoulos, K., & Vosper, L. (1999). VaR without correlations for portfolios of derivative securities. *Journal of Futures Markets*.
- Hull, J. & White, A. (1998). Incorporating Volatility Updating into the Historical Simulation Method for Value-at-Risk.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from archbox.models import GARCH

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 5)

# Carregar dados
data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
returns = data['returns']
n = len(returns)

print(f"S&P 500: {n} observacoes")
print(f"Periodo: {returns.index[0].date()} a {returns.index[-1].date()}")

## 1. VaR Historico

O VaR historico e o metodo mais simples: usa diretamente o **quantil empirico** dos retornos
passados em uma janela movel de $W$ dias (tipicamente $W = 250$, um ano de negociacao):

$$\text{VaR}_{\alpha,t}^{HS} = \text{Quantil}_\alpha(r_{t-W+1}, r_{t-W+2}, \ldots, r_t)$$

**Vantagens:**
- Nao assume distribuicao parametrica
- Captura assimetria e caudas pesadas automaticamente
- Simples de implementar

**Desvantagens:**
- Todas as observacoes na janela tem **peso igual** (independente de quao recentes)
- Sofre de **ghost effects**: um evento extremo influencia o VaR por exatamente $W$ dias
- **Adapta-se lentamente** a mudancas de regime de volatilidade

In [ ]:
# TODO: Calcule VaR historico com janela rolling de 250 dias

window = 250  # 1 ano de negociacao

# Calcular VaR historico rolling
var_hist_95 = np.full(n, np.nan)
var_hist_99 = np.full(n, np.nan)

for t in range(window, n):
    window_returns = returns.values[t - window:t]
    var_hist_95[t] = np.percentile(window_returns, 5)
    var_hist_99[t] = np.percentile(window_returns, 1)

# Violacoes (apenas onde temos VaR calculado)
valid = ~np.isnan(var_hist_95)
viol_95 = (returns.values[valid] < var_hist_95[valid]).sum()
viol_99 = (returns.values[valid] < var_hist_99[valid]).sum()
n_valid = valid.sum()

print(f"=== VaR Historico (janela = {window} dias) ===")
print(f"Observacoes com VaR: {n_valid}")
print(f"Violacoes VaR(95%): {viol_95}/{n_valid} = {viol_95/n_valid:.4f} (esperado: 0.05)")
print(f"Violacoes VaR(99%): {viol_99}/{n_valid} = {viol_99/n_valid:.4f} (esperado: 0.01)")

# Grafico
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(returns.index, returns.values, color='gray', alpha=0.4, linewidth=0.5, label='Retornos')
ax.plot(returns.index, var_hist_95, color='orange', linewidth=1.2, label='VaR Hist. 95%')
ax.plot(returns.index, var_hist_99, color='red', linewidth=1.2, label='VaR Hist. 99%')

# Marcar violacoes
mask_99 = returns.values < var_hist_99
ax.scatter(returns.index[mask_99], returns.values[mask_99],
           color='red', s=15, zorder=5, label='Violacoes 99%')

ax.set_title(f'VaR Historico (janela = {window} dias)')
ax.set_ylabel('Retorno / VaR')
ax.legend(loc='lower left', fontsize=9)
plt.tight_layout()
plt.show()

## 2. Problemas do VaR historico

O VaR historico tem dois problemas principais:

### Ghost effects
Quando um evento extremo entra na janela, o VaR muda abruptamente. Quando esse mesmo evento
sai da janela (apos $W$ dias), o VaR muda novamente. Isso cria "saltos" artificiais.

### Adaptacao lenta
Apos um periodo calmo, a janela nao contem eventos extremos e o VaR e muito otimista.
Quando a volatilidade sobe subitamente, o VaR historico demora $W$ dias para se ajustar.

In [ ]:
# TODO: Mostre caso onde VaR historico falha (apos periodo calmo)

# Identificar periodos de alta e baixa volatilidade
rolling_vol = returns.rolling(window=60).std()

# Encontrar transicao: periodo calmo seguido por alta volatilidade
# Vamos mostrar como o VaR historico reage tardiamente

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Painel 1: Retornos
axes[0].plot(returns.index, returns.values, color='gray', linewidth=0.5)
axes[0].set_title('Retornos S&P 500')
axes[0].set_ylabel('Retorno')

# Painel 2: Volatilidade rolling (60 dias)
axes[1].plot(returns.index, rolling_vol, color='steelblue', linewidth=1)
axes[1].set_title('Volatilidade Rolling (60 dias)')
axes[1].set_ylabel('Desvio padrao')

# Painel 3: VaR historico vs GARCH-Normal
model_g = GARCH(returns.values, p=1, q=1, mean='constant', dist='normal')
res_g = model_g.fit(disp=False)
sigma_g = res_g.conditional_volatility
mu_g = res_g.params[0]
z_99 = stats.norm.ppf(0.01)
var_garch_99 = mu_g + z_99 * sigma_g

axes[2].plot(returns.index, returns.values, color='gray', alpha=0.3, linewidth=0.5, label='Retornos')
axes[2].plot(returns.index, var_hist_99, color='red', linewidth=1.2, label='VaR Hist. 99%')
axes[2].plot(returns.index, var_garch_99, color='green', linewidth=1.2, label='VaR GARCH 99%')
axes[2].set_title('Ghost effect: VaR Historico reage em "degraus", GARCH se adapta suavemente')
axes[2].set_ylabel('VaR')
axes[2].legend(loc='lower left', fontsize=9)

plt.tight_layout()
plt.show()

print("Observe como o VaR historico (vermelho) se move em 'degraus'")
print("enquanto o VaR GARCH (verde) se adapta suavemente a mudancas de volatilidade.")

## 3. Filtered Historical Simulation (FHS)

A **FHS** de Barone-Adesi, Giannopoulos e Vosper (1999) resolve os problemas do VaR historico
combinando um modelo GARCH com bootstrap dos residuos padronizados.

**Ideia central**: os retornos brutos $r_t$ sao nao-estacionarios (clusters de volatilidade),
mas os **residuos padronizados** $z_t = (r_t - \mu_t) / \sigma_t$ sao (aproximadamente) i.i.d.

Procedimento:
1. Estime um modelo GARCH para obter $\hat{\mu}_t$ e $\hat{\sigma}_t$
2. Calcule residuos padronizados: $\hat{z}_t = (r_t - \hat{\mu}_t) / \hat{\sigma}_t$
3. Faca bootstrap (amostragem com reposicao) dos $\hat{z}_t$
4. Para cada $z^*$ amostrado, construa retorno sintetico: $r^* = \hat{\mu}_{T+1} + \hat{\sigma}_{T+1} \cdot z^*$
5. O VaR e o quantil da distribuicao dos $r^*$

A FHS herda a **dinamica temporal** do GARCH (via $\sigma_{T+1}$) e a **forma da distribuicao**
dos dados (via bootstrap dos residuos — sem assumir normalidade).

In [ ]:
# TODO: Implemente FHS: (1) estime GARCH, (2) padronize residuos, (3) bootstrap

# Passo 1: Estimar GARCH(1,1)
model = GARCH(returns.values, p=1, q=1, mean='constant', dist='normal')
results = model.fit(disp=False)

sigma_t = results.conditional_volatility
mu_hat = results.params[0]

print("GARCH(1,1) estimado:")
print(f"  mu = {mu_hat:.6f}")
print(f"  Persistencia = {results.persistence():.4f}")

# Passo 2: Calcular residuos padronizados
z_t = results.resid  # z_t = (r_t - mu) / sigma_t
print("\nResiduos padronizados (z_t):")
print(f"  Media: {z_t.mean():.4f} (esperado: ~0)")
print(f"  Desvio padrao: {z_t.std():.4f} (esperado: ~1)")
print(f"  Assimetria: {pd.Series(z_t).skew():.4f}")
print(f"  Curtose: {pd.Series(z_t).kurtosis():.4f}")

# Visualizar distribuicao dos residuos padronizados
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(z_t, bins=60, density=True, alpha=0.6, color='steelblue', label='z_t empirico')
x = np.linspace(-4, 4, 200)
axes[0].plot(x, stats.norm.pdf(x), 'r-', linewidth=2, label='Normal(0,1)')
axes[0].set_title('Distribuicao dos Residuos Padronizados')
axes[0].set_xlabel('z_t')
axes[0].legend()

stats.probplot(z_t, dist="norm", plot=axes[1])
axes[1].set_title('QQ-Plot: z_t vs Normal')

plt.tight_layout()
plt.show()

print("\nNote as caudas pesadas no QQ-plot — a distribuicao empirica")
print("dos residuos captura essa informacao sem assumir normalidade.")

## 4. FHS passo a passo

Agora vamos implementar o FHS para calcular o VaR dinamico. Para cada dia $t$:

1. Usamos o GARCH para obter $\hat{\sigma}_{t+1}$ (previsao um passo a frente)
2. Fazemos bootstrap dos residuos padronizados: $z^* \sim \hat{F}_z$ (distribuicao empirica)
3. Construimos retornos sinteticos: $r^* = \hat{\mu} + \hat{\sigma}_{t+1} \cdot z^*$
4. $\text{VaR}_\alpha^{FHS} = \text{Quantil}_\alpha(r^*_1, r^*_2, \ldots, r^*_B)$

Equivalentemente (sem bootstrap explicitamente):

$$\text{VaR}_{\alpha,t}^{FHS} = \hat{\mu} + \hat{\sigma}_{t} \cdot \text{Quantil}_\alpha(z_1, z_2, \ldots, z_T)$$

Esta segunda forma e mais eficiente computacionalmente — o quantil dos residuos e fixo,
e apenas $\sigma_t$ varia ao longo do tempo.

In [ ]:
# TODO: z_t = (r_t - mu_t)/sigma_t, bootstrap z*, r* = mu + sigma_{T+1} * z*

# Metodo eficiente: quantil fixo dos residuos * sigma_t variavel
q_z_05 = np.percentile(z_t, 5)   # quantil 5% dos residuos
q_z_01 = np.percentile(z_t, 1)   # quantil 1% dos residuos

print("Quantis dos residuos padronizados:")
print(f"  q(5%)  = {q_z_05:.4f}  (normal: {stats.norm.ppf(0.05):.4f})")
print(f"  q(1%)  = {q_z_01:.4f}  (normal: {stats.norm.ppf(0.01):.4f})")

# VaR FHS dinamico
var_fhs_95 = mu_hat + sigma_t * q_z_05
var_fhs_99 = mu_hat + sigma_t * q_z_01

# Violacoes
viol_fhs_95 = (returns.values < var_fhs_95).sum()
viol_fhs_99 = (returns.values < var_fhs_99).sum()

print("\n=== VaR FHS ===")
print(f"Violacoes VaR(95%): {viol_fhs_95}/{n} = {viol_fhs_95/n:.4f} (esperado: 0.05)")
print(f"Violacoes VaR(99%): {viol_fhs_99}/{n} = {viol_fhs_99/n:.4f} (esperado: 0.01)")

# Metodo com bootstrap explicito (para um unico dia, ilustrativo)
np.random.seed(42)
n_bootstrap = 10000
z_bootstrap = np.random.choice(z_t, size=n_bootstrap, replace=True)

# Simular retornos para o proximo dia
sigma_next = sigma_t[-1]  # volatilidade do ultimo dia
r_bootstrap = mu_hat + sigma_next * z_bootstrap

var_fhs_bootstrap_95 = np.percentile(r_bootstrap, 5)
var_fhs_bootstrap_99 = np.percentile(r_bootstrap, 1)

print(f"\n=== FHS com Bootstrap (proximo dia, {n_bootstrap} simulacoes) ===")
print(f"sigma_{{T+1}} = {sigma_next:.6f}")
print(f"VaR(95%) bootstrap = {var_fhs_bootstrap_95:.6f}")
print(f"VaR(99%) bootstrap = {var_fhs_bootstrap_99:.6f}")
print(f"VaR(95%) analitico = {mu_hat + sigma_next * q_z_05:.6f}")
print(f"VaR(99%) analitico = {mu_hat + sigma_next * q_z_01:.6f}")

## 5. Comparacao: Historico vs FHS vs Parametrico

Vamos comparar os tres metodos lado a lado:

| Metodo | Volatilidade | Distribuicao | Vantagem | Desvantagem |
|--------|-------------|-------------|----------|-------------|
| Historico | Nenhuma (implicita) | Empirica | Simples | Ghost effects, lento |
| Parametrico | GARCH ($\sigma_t$) | Normal/t | Dinamico | Assume distribuicao |
| FHS | GARCH ($\sigma_t$) | Empirica ($z_t$) | Melhor de ambos | Depende do GARCH |

O FHS e geralmente considerado superior pois:
- Adapta-se rapidamente a mudancas de volatilidade (via GARCH)
- Nao assume distribuicao parametrica para as inovacoes
- Captura caudas pesadas e assimetria automaticamente

In [ ]:
# TODO: Compare os 3 metodos com grafico e tabela

# VaR parametrico GARCH-Normal para comparacao
z_95 = stats.norm.ppf(0.05)
z_99 = stats.norm.ppf(0.01)
var_param_95 = mu_hat + z_95 * sigma_t
var_param_99 = mu_hat + z_99 * sigma_t

# Grafico comparativo - VaR 99%
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# Painel superior: VaR 95%
ax = axes[0]
ax.plot(returns.index, returns.values, color='gray', alpha=0.3, linewidth=0.5, label='Retornos')
ax.plot(returns.index, var_hist_95, color='blue', linewidth=1, label='Historico')
ax.plot(returns.index, var_param_95, color='green', linewidth=1, label='GARCH-Normal')
ax.plot(returns.index, var_fhs_95, color='red', linewidth=1, label='FHS')
ax.set_title('VaR 95% - Historico vs Parametrico vs FHS')
ax.set_ylabel('Retorno / VaR')
ax.legend(loc='lower left', fontsize=9)

# Painel inferior: VaR 99%
ax = axes[1]
ax.plot(returns.index, returns.values, color='gray', alpha=0.3, linewidth=0.5, label='Retornos')
ax.plot(returns.index, var_hist_99, color='blue', linewidth=1, label='Historico')
ax.plot(returns.index, var_param_99, color='green', linewidth=1, label='GARCH-Normal')
ax.plot(returns.index, var_fhs_99, color='red', linewidth=1, label='FHS')

# Violacoes do FHS
mask_fhs = returns.values < var_fhs_99
ax.scatter(returns.index[mask_fhs], returns.values[mask_fhs],
           color='darkred', s=15, zorder=5, alpha=0.8, label='Violacoes FHS')

ax.set_title('VaR 99% - Historico vs Parametrico vs FHS')
ax.set_ylabel('Retorno / VaR')
ax.set_xlabel('Data')
ax.legend(loc='lower left', fontsize=9)

plt.tight_layout()
plt.show()

# Tabela resumo
viol_param_95 = (returns.values < var_param_95).sum()
viol_param_99 = (returns.values < var_param_99).sum()

# Para VaR historico, usar apenas periodo com dados
valid_mask = ~np.isnan(var_hist_95)
h95 = (returns.values[valid_mask] < var_hist_95[valid_mask]).sum()
h99 = (returns.values[valid_mask] < var_hist_99[valid_mask]).sum()
n_v = valid_mask.sum()

print("=" * 75)
print(f"{'Metodo':<20} {'Viol 95%':>10} {'Taxa':>8} {'Viol 99%':>10} {'Taxa':>8}")
print("=" * 75)
print(f"{'Historico':<20} {h95:>10} {h95/n_v:>8.4f} {h99:>10} {h99/n_v:>8.4f}")
print(f"{'GARCH-Normal':<20} {viol_param_95:>10} {viol_param_95/n:>8.4f} {viol_param_99:>10} {viol_param_99/n:>8.4f}")
print(f"{'FHS':<20} {viol_fhs_95:>10} {viol_fhs_95/n:>8.4f} {viol_fhs_99:>10} {viol_fhs_99/n:>8.4f}")
print("=" * 75)
print(f"{'Esperado':<20} {'':>10} {'0.0500':>8} {'':>10} {'0.0100':>8}")
print("\nO FHS tipicamente tem melhor calibracao nas caudas (99%)")
print("pois nao assume distribuicao parametrica para os residuos.")